# Generating Ground Truth Data

Search and RAG are impossible to evaluate without a way to score "was this any good?" This notebook builds that yardstick: a **ground truth dataset** of realistic student questions, each labeled with the FAQ record that answers it. We generate it by asking an LLM to emulate a student and write questions from each record — 79 `llm-zoomcamp` FAQ entries become 395 `(question, document_id)` pairs. This dataset is reused for every evaluation in this module: search quality in `02-search-eval.ipynb`, RAG answers in `03-rag-evals.ipynb`, and the LLM judge in `04-llm-judge.ipynb`.

## 1. Loading the FAQ Documents

Same source as modules 01–03: the DataTalks.Club FAQ API, filtered down to the `llm-zoomcamp` course only (79 records out of 1,342 across all courses).

In [ ]:
from zoomcamp.ingest import load_faq_data
documents = load_faq_data()
len(documents)

In [ ]:
documents[10]

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

In [ ]:
documents = documents_llm

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

## 2. Structured Output: Defining the Question Schema

We want the LLM to return a clean list of questions, not free-form text we'd have to parse ourselves. Pydantic's `Questions` model, passed as `text_format` to `responses.parse`, forces the response to match this shape exactly.

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

## 3. The Data-Generation Prompt

The instructions ask the model to **emulate a student**, not summarize the FAQ. Two details matter for data quality: questions should reuse as few words from the record as possible (so search can't win just by keyword-matching the source text), and they should read like something a real person types online — not a formal restatement of the answer.

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json
user_prompt = json.dumps(doc)

In [ ]:
user_prompt

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
response.output_parsed.questions

In [ ]:
doc

## 4. Calling the LLM for Structured Output

One FAQ record, serialized to JSON, becomes the `user` message; the data-generation instructions become the `developer` message. `openai_client.responses.parse(..., text_format=Questions)` returns `response.output_parsed` — already a `Questions` instance, no manual JSON parsing needed. For "I just discovered the course. Can I still join?" the model produces 5 differently-worded variants of that same question.

In [ ]:
from zoomcamp.evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
usage

## 5. Wrapping It in a Reusable Helper (`llm_structured`)

The prompt-building and `responses.parse` call from Section 4 get reused constantly through this whole module, so they're pulled into `zoomcamp/evaluation_utils.py` as `llm_structured(client, instructions, user_prompt, output_type)` — it returns `(parsed_output, usage)`, bundling the token usage right alongside the result so cost can be tracked without extra plumbing.

In [ ]:
from zoomcamp.evaluation_utils import calc_price

In [ ]:
calc_price(usage)

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(records)

## 6. Tracking Cost

`gpt-5.4-mini` is priced per million tokens ($0.75 input / $4.50 output). `calc_price(usage)` turns a `ResponseUsage` object into a cost breakdown for one call — a single question-generation call costs well under a cent, but that adds up over 79 documents, so it's worth watching from the start.

In [ ]:
from zoomcamp.evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

## 7. Retrying on Failures

Structured-output calls occasionally fail transiently (rate limits, flaky JSON). `llm_structured_retry` wraps `llm_structured` with up to 3 attempts and exponential backoff (`2 ** attempt` seconds), re-raising only if every attempt fails. `generate_ground_truth(doc)` uses it to turn one FAQ record into 5 `(question, document)` records plus the call's usage.

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

In [ ]:
ground_truth[:10]

In [ ]:
usages[:10]

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from zoomcamp.evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
ground_truth[10]

## 8. Generating Ground Truth for All Documents

A sequential `tqdm` loop over the first 5 documents proves the pipeline works end-to-end. For all 79 documents, sequential calls would be slow — one LLM round trip per document, one after another — so `map_progress` (a thin wrapper around `ThreadPoolExecutor` that keeps the progress bar in sync with completed futures) fans the calls out across 6 worker threads. 79 documents × 5 questions each gives 395 ground truth records.

In [ ]:
from zoomcamp.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

## 9. Total Cost and Saving the Dataset

Summing `calc_price(usage)["total_cost"]` over all 79 calls (or equivalently, calling `calc_total_price(usages)`) puts the full generation run at roughly **$0.057** — generating an entire evaluation dataset for a few cents. The 395 `(question, document)` pairs are saved to `data/ground_truth.csv`, ready to be loaded by every other notebook in this module.

In [ ]:
from zoomcamp.evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("./data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)